# 07 Time Series and Forecasting — Reference Solutions

Complete solutions for the Songbai Nursing Home Legionnaires' disease outbreak time series exercises.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (prevents Chinese labels from showing as tofu boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

## Question 1: Build a daily hospitalization-count series

In [ ]:
import matplotlib.dates as mdates

# Daily hospitalization counts
hosp_cases = cases[cases["hospitalization_date"].notna()]
hosp_daily = hosp_cases.groupby("hospitalization_date").size()
hosp_daily = hosp_daily.asfreq("D", fill_value=0)
hosp_daily.name = "hospitalizations"

print(f"Series length: {len(hosp_daily)} days")
print(f"Date range: {hosp_daily.index.min().date()} – {hosp_daily.index.max().date()}")
print(f"Total hospitalizations: {hosp_daily.sum()}")

# Add a baseline period
date_range = pd.date_range(
    hosp_daily.index.min() - pd.Timedelta(days=3),
    hosp_daily.index.max() + pd.Timedelta(days=1),
)
hosp_plot = hosp_daily.reindex(date_range, fill_value=0)

# Hospitalization curve + 5-day rolling average
rolling_5 = hosp_daily.rolling(window=5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    hosp_plot.index, hosp_plot.values,
    width=1.0,
    color="#e34a33", edgecolor="white", linewidth=0.5,
    alpha=0.7, label="Daily hospitalizations",
)
ax.plot(rolling_5.index, rolling_5.values, color="navy", linewidth=2,
        label="5-day rolling average")
ax.set_title(
    "Songbai Nursing Home Legionnaires' Disease Daily Hospitalization Curve + 5-day Rolling Average, January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Hospitalization")
ax.set_ylabel("Number of Hospitalizations")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## Question 2: Hospitalization forecasting and window comparison

In [ ]:
# Window comparison
print("=== Hospitalization forecast MAE ===")
best_w, best_mae = 3, float("inf")

for w in [3, 5, 7]:
    pred_w = hosp_daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = hosp_daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w

print(f"\n→ Best window: window={best_w} (MAE={best_mae:.3f})")

# Actual vs Predicted
pred_best = hosp_daily.rolling(window=best_w).mean().shift(1).dropna()
actual_best = hosp_daily.loc[pred_best.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(actual_best.index, actual_best.values, marker="o", markersize=4,
        label="Actual hospitalizations", color="#e34a33")
ax.plot(pred_best.index, pred_best.values, marker="s", markersize=4,
        label=f"Predicted ({best_w}-day MA)", color="navy", linestyle="--")
ax.set_title(
    f"Songbai Nursing Home Hospitalizations Actual vs Predicted ({best_w}-day Rolling Average), January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date")
ax.set_ylabel("Number of Hospitalizations")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## Question 3 (challenge): Epidemic curves grouped by severity

In [ ]:
# Build daily onset-count series by severity
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

# Build daily series (all severities share the date range, including baseline period)
all_onset = cases.groupby("symptom_onset_date").size()
all_onset = all_onset.asfreq("D", fill_value=0)

# Add a baseline period
date_range = pd.date_range(
    all_onset.index.min() - pd.Timedelta(days=3),
    all_onset.index.max() + pd.Timedelta(days=1),
)

severity_daily = {}
for sev in severity_levels:
    sub = cases[cases["clinical_severity"] == sev]
    s = sub.groupby("symptom_onset_date").size()
    severity_daily[sev] = s.reindex(date_range, fill_value=0)

sev_df = pd.DataFrame(severity_daily)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(sev_df))

for sev in severity_levels:
    ax.bar(
        sev_df.index, sev_df[sev].values, bottom=bottom,
        width=1.0,
        color=colors[sev], edgecolor="white", linewidth=0.5,
        alpha=0.8, label=sev,
    )
    bottom += sev_df[sev].values

ax.set_title(
    "Songbai Nursing Home Legionnaires' Disease Epidemic Curve (Stratified by Severity), January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

# Peak day for each severity
print("=== Peak day by severity ===")
for sev in severity_levels:
    peak_date = sev_df[sev].idxmax()
    peak_count = sev_df[sev].max()
    print(f"  {sev:10s}  peak day = {peak_date.date()}  {peak_count} people that day")

print("\n→ Observe whether severe cases appear in sync with mild ones, or with a time lag")
print("→ If severe cases cluster in the middle of the outbreak, it may mean residents with higher exposure doses fell ill later")

### Interpretation

- **Hospitalization curve**: the hospitalization peak lags the onset peak by a few days; this lag can be used to forecast bed demand
- **Window choice**: a smaller window (3 days) usually performs better in acute clusters because case counts change quickly
- **Severity stratification**: if severe cases concentrate in a particular time window, it may hint at a specific exposure event or a high-risk group
- **Limitation**: the rolling average is the simplest baseline model and cannot capture trend turning points; more advanced methods (such as ARIMA) can improve on this foundation

## Question 4 Solution

In [ ]:
import numpy as np

# --- Data: 2-year Enterovirus report line list ---
rng = np.random.default_rng(701)
start = pd.Timestamp("2024-01-01")
n_days = 730
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.array([d.dayofyear for d in dates_all]) % 366

# Two seasonal peaks: early summer (around April) and the start of school (around Sept-Oct)
peak1 = np.exp(-0.5 * ((doy - 100) / 22) ** 2)
peak2 = np.exp(-0.5 * ((doy - 270) / 28) ** 2)
weights = 0.05 + peak1 + 0.7 * peak2
probs = weights / weights.sum()

n_cases = 480
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "age_group": rng.choice(["<5", "5-9", "10-14"], size=n_cases, p=[0.55, 0.30, 0.15]),
}).sort_values("report_date").reset_index(drop=True)

# Daily -> weekly report counts (fill in missing weeks)
daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
weekly = daily.resample("W").sum()
weekly.name = "cases"
print(f"Series length: {len(weekly)} weeks | Total reports: {weekly.sum()}")

# Average weekly report count by month
monthly_avg = weekly.groupby(weekly.index.month).mean().sort_values(ascending=False)
print("\n=== Average weekly report count by month (top 5) ===")
print(monthly_avg.head().round(2))

# Week-over-week change
wow_change = (weekly - weekly.shift(1)).dropna()
print(f"\nWeek-over-week change: mean {wow_change.mean():+.2f}, largest single-week increase {wow_change.max():.0f}")

# Weekly epidemic curve, marking the top 8 peak weeks
top_weeks = weekly.sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(weekly.index, weekly.values, width=5, color="#6A9BCC",
       edgecolor="white", alpha=0.75, label="Weekly reports")
ax.scatter(top_weeks.index, top_weeks.values, color="#D97757", s=45,
           zorder=5, label="Top 8 peak weeks")
ax.set_title("Weekly Enterovirus Reports (2024-2025)", fontweight="bold")
ax.set_xlabel("Week")
ax.set_ylabel("Number of Reports")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\n→ Report counts show a clear bimodal seasonal pattern: peaking around April (early summer) and September-October (start of the school year) each year")
print("→ This may be related to increased close contact among children in schools, after-school care centers, etc., since enterovirus spreads via the fecal-oral route and respiratory droplets")

## Question 5 Solution

In [ ]:
import numpy as np

# --- Data: 120-day Influenza report line list ---
rng = np.random.default_rng(707)
start = pd.Timestamp("2025-11-01")
n_days = 120
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# A single flu season: peaks around the midpoint (around day 60), plus a pattern of lower weekend reporting
season_shape = np.exp(-0.5 * ((day_idx - 60) / 18) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.45)  # fewer visits/reports on weekends
weights = (0.08 + season_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 560
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"Series length: {len(daily)} days | Total reports: {daily.sum()}")

# Stationarity test
from statsmodels.tsa.stattools import adfuller
adf_stat, p_value, *_ = adfuller(daily)
print(f"\nADF statistic = {adf_stat:.3f}, p-value = {p_value:.3f}")
print("→ p >= 0.05: the series is non-stationary (a clear seasonal trend), so differencing is needed (d, D >= 1)")

# Split training / test set: last 14 days as the test set
train, test = daily.iloc[:-14], daily.iloc[-14:]
print(f"\nTraining set: {len(train)} days, test set: {len(test)} days")

# SARIMA(1,1,1)(1,1,0,7) -- period = 7 days
from statsmodels.tsa.statespace.sarimax import SARIMAX
model_sarima = SARIMAX(
    train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 7),
).fit(disp=False)
forecast_sarima = model_sarima.forecast(steps=14)
mae_sarima = mean_absolute_error(test.values, forecast_sarima.values)
print(f"\nSARIMA(1,1,1)(1,1,0,7)：MAE={mae_sarima:.3f}，AIC={model_sarima.aic:.2f}")

# Baseline: last 7-day rolling average of the training set (treated as a constant forecast for the next 14 days)
baseline_val = train.rolling(7).mean().iloc[-1]
baseline_pred = pd.Series(baseline_val, index=test.index)
mae_baseline = mean_absolute_error(test.values, baseline_pred.values)
print(f"7-day rolling average baseline: MAE={mae_baseline:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index[-30:], train.values[-30:], color="#6B6B6B",
        linewidth=1.2, label="Training (last 30 days)")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=4, label="Actual")
ax.plot(test.index, forecast_sarima.values, color="#D97757", linewidth=1.8,
        marker="^", markersize=4, linestyle="--",
        label=f"SARIMA (MAE={mae_sarima:.2f})")
ax.plot(test.index, baseline_pred.values, color="#6A9BCC", linewidth=1.8,
        linestyle=":", label=f"7-day rolling average (MAE={mae_baseline:.2f})")
ax.set_title("Daily Influenza Reports: SARIMA vs Rolling-Average Baseline", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Reports")
ax.legend()
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

if mae_sarima < mae_baseline:
    print("\n→ SARIMA outperforms the rolling-average baseline: because SARIMA additionally captures the 7-day cycle of lower weekend reporting")
else:
    print("\n→ In this run the rolling-average baseline performs close to or better than SARIMA, showing that a simple baseline still has value when the data volume is small")
print("→ Regardless of which model is more accurate, both need >= 1 full cycle of data to learn the within-week pattern")

## Question 6 Solution

In [ ]:
import numpy as np

# --- Data: 90-day COVID-19 report line list ---
rng = np.random.default_rng(719)
start = pd.Timestamp("2026-03-01")
n_days = 90
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# A single wave: rises then falls, peaking around day 40, with "lower weekend reporting" noise layered on top
wave_shape = np.exp(-0.5 * ((day_idx - 40) / 14) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.5)
weights = (0.05 + wave_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 600
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"Series length: {len(daily)} days | Total confirmed cases: {daily.sum()}")

# Rolling-average window comparison
print("\n=== Rolling-average window MAE comparison ===")
best_w, best_mae = 3, float("inf")
for w in [3, 7, 14]:
    pred_w = daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w:>2d}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w
print(f"\n→ Best window: window={best_w} (MAE={best_mae:.3f})")

# Peak-day comparison
roll7 = daily.rolling(7, min_periods=1).mean()
peak_raw = daily.idxmax()
peak_smooth = roll7.idxmax()
print(f"\nRaw daily count peak day: {peak_raw.date()} ({daily.max()} cases)")
print(f"7-day rolling average peak day: {peak_smooth.date()} ({roll7.max():.1f} cases)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0, color="#6A9BCC",
       edgecolor="white", alpha=0.6, label="Daily new cases")
ax.plot(roll7.index, roll7.values, color="#D97757", linewidth=2,
        label="7-day rolling average")
ax.axvline(peak_raw, color="#6A9BCC", linestyle=":", linewidth=1.5)
ax.axvline(peak_smooth, color="#D97757", linestyle=":", linewidth=1.5)
ax.set_title("COVID-19 Daily New Cases and 7-day Rolling Average", fontweight="bold")
ax.set_xlabel("Report Date")
ax.set_ylabel("Daily New Cases")
ax.legend()
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\n→ Raw daily new-case counts swing up and down due to 'weekend reporting delays,' making it easy to misjudge peaks or turning points")
print("→ Rolling averages smooth out the noise and reveal the true trend, which is the most common presentation used on public health surveillance dashboards")

## Question 7 Solution

In [ ]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# --- Data: one-year Dengue report line list ---
rng = np.random.default_rng(709)
start = pd.Timestamp("2025-01-01")
n_days = 365
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.arange(n_days)

# Summer peak: roughly June-September (mosquito vector density rises with temperature and humidity)
summer_shape = np.exp(-0.5 * ((doy - 210) / 35) ** 2)
weights = 0.05 + summer_shape
probs = weights / weights.sum()

n_cases = 520
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "township": rng.choice(["A區", "B區", "C區"], size=n_cases, p=[0.5, 0.3, 0.2]),
}).sort_values("report_date").reset_index(drop=True)

daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"Series length: {len(daily)} days | Total reports: {daily.sum()}")

# Lag features
ts = daily.to_frame("cases").reset_index(names="date")
ts["day_idx"] = range(len(ts))
ts["lag_1"] = ts["cases"].shift(1)
ts["lag_2"] = ts["cases"].shift(2)
ts_model = ts.dropna().reset_index(drop=True)
print(f"Usable rows: {len(ts_model)}")

# dispersion ratio
disp = ts_model["cases"].var() / ts_model["cases"].mean()
print(f"\ndispersion = variance / mean = {disp:.2f}")
print("→ > 1.5 is considered overdispersed -> switch to Negative Binomial\n")

# Poisson regression
model_pois = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx", data=ts_model, family=sm.families.Poisson(),
).fit()
pred_pois = model_pois.predict(ts_model)
mae_pois = mean_absolute_error(ts_model["cases"], pred_pois)
print(f"Poisson + lag: MAE={mae_pois:.3f}, AIC={model_pois.aic:.2f}")

# Negative Binomial regression
model_nb = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx", data=ts_model,
    family=sm.families.NegativeBinomial(alpha=1.0),
).fit()
pred_nb = model_nb.predict(ts_model)
mae_nb = mean_absolute_error(ts_model["cases"], pred_nb)
print(f"Negative Binomial + lag: MAE={mae_nb:.3f}, AIC={model_nb.aic:.2f}")

# IRR
irr_table = pd.DataFrame({
    "coef (log scale)": model_nb.params,
    "IRR exp(coef)": np.exp(model_nb.params),
})
print("\n=== Negative Binomial coefficient table (IRR) ===")
print(irr_table.round(3))

lag1_irr = np.exp(model_nb.params["lag_1"])
print(f"\n→ IRR for lag_1 = {lag1_irr:.3f}: for each additional case the previous day, the next day's expected report count becomes {lag1_irr:.2f}x")
print(f"→ dispersion={disp:.2f} is clearly > 1.5, and the Negative Binomial's AIC ({model_nb.aic:.1f})")
print(f"  is lower than the Poisson's ({model_pois.aic:.1f}), indicating NB is a better fit for this clustered, overdispersed vector-borne transmission data")
print("→ The dengue summer peak is related to temperature and rainfall raising the density and biting frequency of the mosquito vectors (Aedes aegypti / Aedes albopictus)")

## Question 8 Solution

In [ ]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.arima.model import ARIMA

# --- Data: Songbai Nursing Home "recontamination" extended scenario (first wave + quiet period + second wave) ---
rng = np.random.default_rng(713)
start = pd.Timestamp("2026-02-01")

# First wave: a 21-day outbreak similar to the main dataset, peaking around day 8
wave1_days = 21
wave1_shape = np.exp(-0.5 * ((np.arange(wave1_days) - 8) / 3.5) ** 2)
wave1_dates = pd.date_range(start, periods=wave1_days, freq="D")

# Second wave: the hot-water system is recontaminated 55 days later, smaller in scale and more concentrated
wave2_start = start + pd.Timedelta(days=55)
wave2_days = 15
wave2_shape = np.exp(-0.5 * ((np.arange(wave2_days) - 6) / 3) ** 2)
wave2_dates = pd.date_range(wave2_start, periods=wave2_days, freq="D")

def _sample_wave(dates_wave, shape, n, rng):
    probs = shape / shape.sum()
    idx = rng.choice(len(dates_wave), size=n, p=probs)
    return dates_wave[idx]

onset1 = _sample_wave(wave1_dates, wave1_shape, 92, rng)
onset2 = _sample_wave(wave2_dates, wave2_shape, 34, rng)
onset_all = np.concatenate([onset1, onset2])

line_list = pd.DataFrame({
    "case_id": np.arange(1, len(onset_all) + 1),
    "symptom_onset_date": onset_all,
}).sort_values("symptom_onset_date").reset_index(drop=True)
print(f"Total cases (both waves combined): {len(line_list)}")

# Complete daily onset-count series
full_range = pd.date_range(wave1_dates.min(), wave2_dates.max(), freq="D")
daily = line_list.groupby("symptom_onset_date").size()
daily = daily.reindex(full_range, fill_value=0)
daily.name = "cases"
print(f"Series length: {len(daily)} days ({daily.index.min().date()} ~ {daily.index.max().date()})")

# Training set = wave 1 + quiet period (first 51 days); test set = day 52 onward (covers wave 2)
cutoff = wave1_days + 30
train, test = daily.iloc[:cutoff], daily.iloc[cutoff:]
print(f"Training set: {len(train)} days, test set: {len(test)} days (wave 2 falls within the test set)")

# (a) 3-day rolling average, using the training set's last value as a fixed forecast for the future
roll_val = train.rolling(3, min_periods=1).mean().iloc[-1]
pred_roll = pd.Series(roll_val, index=test.index)
mae_roll = mean_absolute_error(test.values, pred_roll.values)

# (b) Poisson regression + lag, iterative prediction (use the previous step's prediction as the next step's lag)
ts = train.to_frame("cases").reset_index(names="date")
ts["day_idx"] = range(len(ts))
ts["lag_1"] = ts["cases"].shift(1)
ts["lag_2"] = ts["cases"].shift(2)
ts_model = ts.dropna().reset_index(drop=True)
model_pois = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx", data=ts_model, family=sm.families.Poisson(),
).fit()

history = list(train.values[-2:])
pred_pois = []
for i in range(len(test)):
    row = pd.DataFrame({
        "lag_1": [history[-1]], "lag_2": [history[-2]], "day_idx": [len(train) + i],
    })
    p = model_pois.predict(row).iloc[0]
    pred_pois.append(p)
    history.append(p)
mae_pois = mean_absolute_error(test.values, pred_pois)

# (c) ARIMA(1,1,1)
model_arima = ARIMA(train, order=(1, 1, 1)).fit()
forecast_arima = model_arima.forecast(steps=len(test))
mae_arima = mean_absolute_error(test.values, forecast_arima.values)

print("\n=== Three-model MAE comparison (test period covers wave 2) ===")
print(f"  (a) 3-day rolling average (fixed value)   MAE={mae_roll:.3f}")
print(f"  (b) Poisson + lag (iterative prediction)  MAE={mae_pois:.3f}")
print(f"  (c) ARIMA(1,1,1)                          MAE={mae_arima:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index, train.values, color="#6B6B6B", linewidth=1.2, label="Training period (wave 1 + quiet period)")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=4, label="Actual (includes wave 2)")
ax.plot(test.index, pred_roll.values, color="#6A9BCC", linestyle=":", linewidth=1.8,
        label=f"3-day rolling average (MAE={mae_roll:.2f})")
ax.plot(test.index, pred_pois, color="#788C5D", linestyle="--", linewidth=1.8,
        label=f"Poisson + lag (MAE={mae_pois:.2f})")
ax.plot(test.index, forecast_arima.values, color="#D97757", linestyle="--", linewidth=1.8,
        label=f"ARIMA(1,1,1) (MAE={mae_arima:.2f})")
ax.set_title("Songbai Nursing Home Recontamination: Three-model Forecasts vs Actual Onset Counts", fontweight="bold")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Daily Onset Count")
ax.legend(fontsize=8)
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\n→ In the test period, all three models predict close to 0 for nearly the entire time, completely failing to anticipate wave 2")
print("→ Because all three models only extrapolate from the historical pattern of 'wave 1 + quiet period,'")
print("  and wave 2 is a brand-new exposure event (the hot-water system was recontaminated), not a continuation of an existing trend")
print("→ Implication: statistical models can only 'extrapolate past patterns' and cannot predict a brand-new exposure source;")
print("  real-time surveillance systems still need to be paired with active measures like environmental sampling and water-quality monitoring to detect recontamination early")

## Question 9 Solution

In [ ]:
import numpy as np

# --- Data: 160-day daily influenza-like illness (ILI) report series (with trend, weekly cycle, holiday effect) ---
rng = np.random.default_rng(715)
start = pd.Timestamp("2025-11-01")
n_days = 160
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# Long-term upward trend: ILI reports climb week over week after winter sets in
trend = 35 + 0.35 * day_idx

# Weekly cycle: more clinic visits/reports on weekdays, fewer on weekends
dow = dates_all.dayofweek
weekly_pattern = np.where(dow < 5, 1.2, 0.7)

# Holiday effect: after the New Year holiday (falls in the training period) and
# the Tomb-Sweeping Day holiday (falls in the test period), family gatherings and
# travel increase contact, driving a surge in visits
holiday_dates = pd.to_datetime([
    "2026-01-02", "2026-01-03", "2026-01-04",   # after the New Year holiday
    "2026-04-04", "2026-04-05", "2026-04-06",   # after the Tomb-Sweeping Day holiday
])
holiday_boost = np.where(np.isin(dates_all, holiday_dates), 1.8, 1.0)

lam = trend * weekly_pattern * holiday_boost
y = rng.poisson(lam).astype(float)

ili = pd.DataFrame({"ds": dates_all, "y": y})

print(f"Series length: {len(ili)} days")
print(f"Date range: {ili['ds'].min().date()} - {ili['ds'].max().date()}")
print(f"Report count range: {ili['y'].min():.0f} - {ili['y'].max():.0f} (mean {ili['y'].mean():.1f})")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(ili["ds"], ili["y"], width=1.0, color="#6A9BCC", edgecolor="white",
       alpha=0.75, label="Daily reports")
for d in holiday_dates:
    ax.axvline(d, color="#D97757", linestyle=":", linewidth=1, alpha=0.7)
ax.set_title("Daily ILI Reports, 2025-11 to 2026-04 (orange lines = post-holiday peak days)", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Daily report count")
ax.legend()
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
import logging

logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
from prophet import Prophet

# Train / test split: last 14 days as the test set
train, test = ili.iloc[:-14].copy(), ili.iloc[-14:].copy()
print(f"Training set: {len(train)} days, test set: {len(test)} days")

# Basic Prophet model (no holidays yet)
model = Prophet(
    weekly_seasonality=True, yearly_seasonality=False,
    daily_seasonality=False, interval_width=0.9,
)
model.fit(train)

future = model.make_future_dataframe(periods=14)
forecast = model.predict(future)
forecast_test = forecast.set_index("ds").loc[test["ds"], "yhat"]
mae_prophet = mean_absolute_error(test["y"].values, forecast_test.values)

# Naive persistence baseline: use the previous day's actual value as today's prediction
persistence_pred = ili["y"].shift(1).iloc[-14:]
mae_persistence = mean_absolute_error(test["y"].values, persistence_pred.values)

print(f"\nProphet (no holidays): MAE={mae_prophet:.3f}")
print(f"Naive persistence baseline: MAE={mae_persistence:.3f}")

# Read off the trend / weekly components
fig_comp = model.plot_components(forecast)
plt.show()

# Add the holiday effect: the visit surge after the New Year and Tomb-Sweeping Day holidays
holidays_df = pd.DataFrame({
    "holiday": "post_holiday_surge",
    "ds": pd.to_datetime([
        "2026-01-02", "2026-01-03", "2026-01-04",
        "2026-04-04", "2026-04-05", "2026-04-06",
    ]),
    "lower_window": 0,
    "upper_window": 0,
})

model_hol = Prophet(
    weekly_seasonality=True, yearly_seasonality=False,
    daily_seasonality=False, interval_width=0.9,
    holidays=holidays_df,
)
model_hol.fit(train)
future_hol = model_hol.make_future_dataframe(periods=14)
forecast_hol = model_hol.predict(future_hol)
forecast_hol_test = forecast_hol.set_index("ds").loc[test["ds"], "yhat"]
mae_prophet_hol = mean_absolute_error(test["y"].values, forecast_hol_test.values)
print(f"Prophet (with holiday effect): MAE={mae_prophet_hol:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train["ds"].iloc[-30:], train["y"].iloc[-30:], color="#6B6B6B",
        linewidth=1.2, label="Training (last 30 days)")
ax.plot(test["ds"], test["y"], color="#1A1A1A", linewidth=2,
        marker="o", markersize=4, label="Actual")
ax.plot(test["ds"], forecast_test.values, color="#6A9BCC", linewidth=1.8,
        linestyle="--", label=f"Prophet no holidays (MAE={mae_prophet:.2f})")
ax.plot(test["ds"], forecast_hol_test.values, color="#D97757", linewidth=1.8,
        linestyle="--", label=f"Prophet with holidays (MAE={mae_prophet_hol:.2f})")
ax.set_title("ILI Reports: Prophet Forecast vs. Actual (with vs. without holiday effect)", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Daily report count")
ax.legend(fontsize=8)
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\n→ Prophet (no holidays) MAE={mae_prophet:.3f} vs naive persistence MAE={mae_persistence:.3f}")
if mae_prophet < mae_persistence:
    print("  Prophet effectively captured the long-term trend and weekly cycle, beating the naive baseline of simply using yesterday's value to predict today")
else:
    print("  At this data volume, naive persistence performs close to or better than Prophet, showing that a simple baseline still has value")
print(f"→ After adding the New Year / Tomb-Sweeping Day holiday effect, MAE changed from {mae_prophet:.3f} to {mae_prophet_hol:.3f}")
print("→ ILI reports commonly show a pattern where holiday family gatherings and travel increase contact, driving a surge in visits after the holiday;")
print("  a model that ignores the holiday effect tends to underestimate report counts in the days following a holiday. Prophet's holidays parameter lets you explicitly model these known events")